# Downlink CSI Feedback Using a Single Multi-Port CSI-RS

This example demonstrates the simplest codebook-based downlink CSI feedback procedure in NeoRadium. A gNB periodically transmits one non-zero-power, multi-port CSI reference signal (NZP CSI-RS) without applying an additional directional beam. The UE uses the received CSI-RS to estimate the downlink MIMO channel and report:

- **Rank Indicator (RI):** the recommended number of transmission layers;
- **Precoding Matrix Indicator (PMI):** the preferred Type-I single-panel codebook precoder; and
- **Channel Quality Indicator (CQI):** the recommended modulation and code-rate operating point.

The gNB uses the reported RI, PMI, and CQI to configure and precode subsequent PDSCH transmissions. Because no separate CSI-RS beam sweep or CRI selection is performed, the Type-I codebook precoder provides the complete spatial beamforming operation. This represents a fully digital precoding example and serves as a foundation for the more advanced beam-management examples that use multiple directional CSI-RS resources or hierarchical beam sweeping and probing.

The simulation includes:

1. a dual-polarized MIMO channel based on a clustered delay line model;
2. periodic transmission of a single multi-port CSI-RS resource;
3. UE-side RI, PMI, and CQI calculation;
4. gNB-side PDSCH reconfiguration based on the reported CSI;
5. Type-I codebook precoding of the PDSCH; and
6. receiver equalization, LDPC decoding, and transport-block CRC verification.

The CSI feedback is delayed in the sense that a report generated from one CSI-RS occasion is applied to subsequent PDSCH slots. The example can use either wideband PMI or subband PMI, depending on the configured precoding resource-group size. Setting `prgSize = 0` selects wideband PMI only.

In [1]:
import numpy as np

from neoradium import BandwidthPart, PDSCH, AntennaPanel, CdlChannel, random
from neoradium import CsiRsConfig, CsiRsSet, CsiRs, CsiReport, CsiReportMan
from neoradium.utils import toLinear

In [2]:
numSlots = 100                          # Number of slots in the communication loop
snrDb = -10                             # SNR in dB
random.setSeed(1234)                    # Make results reproducible
prgSize = 4                             # Subband size (Set to 0 for wideband PMI)

# Create a bandwidth part with 24 resource blocks and 15 kHz subcarrier spacing
bwp = BandwidthPart(numRbs=24, spacing=15)       

# Create a CDL channel model
channel = CdlChannel(bwp, profile='C', delaySpread=30, carrierFreq=4e9, dopplerShift=5,
                     txAntenna=AntennaPanel([2,4], polarization='x'),   # 16 TX antennas
                     rxAntenna=AntennaPanel([1,2], polarization='x'),   # 4 RX antennas
                     rxOrientation = [180,0,0])

# One multi-port CSI-RS
pmiSet = CsiRsSet("NZP", bwp, resourceType="periodic", rsId=1, period=20*(bwp.u+1), 
                  csiRsList=[CsiRs(resourceId=1, symbols=[4], numPorts=channel.txAntenna.numPorts,
                                   freqMap="001111", cdmSize=4)])
csiRsConfig = CsiRsConfig([pmiSet])
# csiRsConfig.print()                   # Uncomment to print CSI-RS configuration details

pmiRep = CsiReport(pmiSet, reportId=pmiSet.rsId+10, quantity="RiPmiCqi", reportType="periodic",
                   period=20*(bwp.u+1), offset=5, prgSize=prgSize, allowedRanks=[1,2],
                   txAntenna=channel.txAntenna, rxAntenna=channel.rxAntenna)
csiReportMan = CsiReportMan([pmiRep])
# csiReportMan.print()                 # Uncomment to print CSI report configuration details

# The PDSCH object is created once we have the first RI/PMI/CQI feedback and 
# recreated later if CQI changes.
pdsch = None
precoder = None
failedSlots = 0
for slotNo in range(numSlots):
    channelMatrix = channel.getChannelMatrix()

    # Retrieve CSI feedback generated from previous CSI-RS occasions
    csiReportInfo = csiReportMan.getFeedback()              # Get all available CSI reports from CsiReport objects
    for reportId, csiFeedback in csiReportInfo.items():     # Get the CSI feedback for each report
        if reportId == pmiRep.reportId:                     # RI/PMI/CQI report
            print(f"Slot {slotNo}: Received RI/PMI/CQI (ReportID: {reportId})")
            print(f"  RI: {csiFeedback.ri.ri} (Score:{csiFeedback.ri.score:.3f})")
            print(f"  WB PMI: {csiFeedback.pmi.wbPMI}")
            print(f"  WB precoder shape: {csiFeedback.pmi.wbW.shape}")
            if csiFeedback.pmi.sbWs is not None:
                print(f"  {len(csiFeedback.pmi.sbWs)} SB precoders: ")
                for i, (rbIdx, w) in enumerate(csiFeedback.pmi.sbWs): 
                    print(f"    RBs: {str(rbIdx):<20} precoder shape: {str(w.shape):<10} PMI: {csiFeedback.pmi.sbPMIs[i]}")  
            if "cqi" in pmiRep.quantity.lower():
                print(f"  CQI: {csiFeedback.cqi.cqi}")
                modulation, coderateX1024 = pmiRep.getModRate(csiFeedback.cqi.cqi)
                print(f"  Modulation: {modulation}")
                print(f"  Coderate: {coderateX1024}/1024")
                print(f"  CQI BLER: {csiFeedback.cqi.bler:.2f} %")

            precoder = csiFeedback.pmi.wbW if csiFeedback.pmi.sbWs is None else csiFeedback.pmi.sbWs   
            if pdsch is None:   
                # First RI/PMI/CQI feedback -> create PDSCH and LDPC codec objects
                print(f"Slot {slotNo}: Starting PDSCH (Mod:{modulation}, "
                      f"Coderate:{coderateX1024}/1024)")
                pdsch = PDSCH(bwp, numLayers=csiFeedback.ri.ri, csiRsConfig=csiRsConfig, 
                              modulation=modulation, prgSize=pmiRep.prgSize)
                pdsch.setDMRS(additionalPos=2)
                ldpc = pdsch.getLdpcCodec(coderates = coderateX1024/1024)
                
            elif ( (pdsch.modems[0].modulation != modulation) or 
                   (pdsch.numLayers != csiFeedback.ri.ri) ):
                # Modulation or number of layers changed -> Recreate PDSCH and LDPC codec objects
                print(f"Slot {slotNo}: CQI/RI changed -> Mod:{modulation}, "
                      f"Coderate:{coderateX1024}/1024")
                pdsch = PDSCH(bwp, numLayers=csiFeedback.ri.ri, csiRsConfig=csiRsConfig, 
                              modulation=modulation, prgSize=pmiRep.prgSize)
                pdsch.setDMRS(additionalPos=2)
                ldpc = pdsch.getLdpcCodec(coderates = coderateX1024/1024)
            
            elif ldpc.coderates[0] != (coderateX1024/1024):
                # Coderate changed -> Recreate the LDPC codec object only
                print(f"Slot {slotNo}: CQI changed -> Coderate:{coderateX1024}/1024")
                ldpc = pdsch.getLdpcCodec(coderates = coderateX1024/1024)
        else:
            print(f"Unknown report: {reportId}")

    # Create a transmitted resource grid.
    txGrid = bwp.createGrid(channel.txAntenna.numEl)
    if pdsch is not None:
        # Create random data, LDPC encode it, and put it in the PDSCH's internal resource grid.
        # Then precode the PDSCH into the transmitted resource grid - txGrid.
        pdsch.initGrid()
        numBits = pdsch.getBitCapacity()[0]
        txBlock = random.bits(ldpc.txBlockSizes[0])
        rateMatchedCodeBlocks = ldpc.encode(txBlock, numBits)  
        pdsch.setPdschData(rateMatchedCodeBlocks)
        pdsch.precodeTo(txGrid, precoder)
    
    # Map any CSI-RS resources scheduled in the current slot
    csiRsResources = csiRsConfig.getResources()
    for csiSetId, setResources in csiRsResources.items():
        if csiSetId == pmiSet.rsId:                             # CSI-RS for RI/PMI/CQI:
            print(f"Slot {slotNo}: Sending CSI resources for RI/PMI/CQI (Set ID:{pmiSet.rsId})")
            for resourceId, (lIdx, kIdx, pmiReValues) in setResources.items():
                # Simulation note:
                # Scale the CSI-RS to keep its aggregate transmit power approximately
                # consistent with the PDSCH. This avoids unintentionally reducing the
                # effective PDSCH SNR in this simulation, where the noise variance is
                # derived from the average received signal power. This is a simulation
                # convenience only; it is **NOT** a 3GPP requirement or recommendation
                # and is not representative of how practical systems necessarily
                # implement CSI-RS transmission.
                csiRs = csiRsConfig.getById(csiSetId, resourceId)
                pf = np.sqrt( csiRs.numPorts/(channel.txAntenna.numEl*csiRs.cdmSize))  # Power factor

                # pmiReValues is an nt x numCsiRsRE matrix.
                txGrid[:, lIdx, kIdx] = (pmiReValues*pf, "CSIRS_NZP", resourceId)
    
    # Apply the channel model and add AWGN noise
    rxGrid = txGrid.applyChannel(channelMatrix) 
    noisyRxGrid = rxGrid.addNoise(snrDb=snrDb)            # Add noise

    # UE processing of the received resource grid to generate reports
    csiReportMan.processRxGrid(noisyRxGrid, csiRsResources)

    if pdsch is not None:
        # Receiver side processing of the PDSCH: equalization and LDPC decoding
        effChannelMatrix = channel.getEffChannel(channelMatrix, precoder)
        eqGrid, llrScales = pdsch.equalize(noisyRxGrid, effChannelMatrix)
        llrs = pdsch.getLLRs(eqGrid, llrScales)
        decodedTxBlocks, crcMatch = ldpc.decode(llrs)
        print(f"Slot {slotNo}: TxBlock CRC Match: {crcMatch[0][0]}")
        failedSlots += 1-int(crcMatch[0][0])

    # Go to the next channel instance for the next slot
    channel.goNext()

print(f"{failedSlots} of {numSlots} slots failed.")

Slot 0: Sending CSI resources for RI/PMI/CQI (Set ID:1)
Slot 7: Received RI/PMI/CQI (ReportID: 11)
  RI: 1 (Score:2.226)
  WB PMI: (I1:[0, 0, 0], I2:2)
  WB precoder shape: (16, 1)
  6 SB precoders: 
    RBs: [0, 1, 2, 3]         precoder shape: (16, 1)    PMI: (I1:[0, 0, 0], I2:2)
    RBs: [4, 5, 6, 7]         precoder shape: (16, 1)    PMI: (I1:[0, 0, 0], I2:1)
    RBs: [8, 9, 10, 11]       precoder shape: (16, 1)    PMI: (I1:[0, 0, 0], I2:2)
    RBs: [12, 13, 14, 15]     precoder shape: (16, 1)    PMI: (I1:[0, 0, 0], I2:2)
    RBs: [16, 17, 18, 19]     precoder shape: (16, 1)    PMI: (I1:[0, 0, 0], I2:2)
    RBs: [20, 21, 22, 23]     precoder shape: (16, 1)    PMI: (I1:[0, 0, 0], I2:2)
  CQI: 7
  Modulation: 16QAM
  Coderate: 378/1024
  CQI BLER: 7.95 %
Slot 7: Starting PDSCH (Mod:16QAM, Coderate:378/1024)
Slot 7: TxBlock CRC Match: True
Slot 8: TxBlock CRC Match: True
Slot 9: TxBlock CRC Match: True
Slot 10: TxBlock CRC Match: True
Slot 11: TxBlock CRC Match: True
Slot 12: TxBlock 